# Heritage Twin MVP - Pipeline gộp: Video -> COLMAP -> 3D Gaussian Splatting

Notebook này gộp 2 notebook cũ (`video_to_colmap` + `train_from_colmap_zip`) thành 1 pipeline liền mạch:

```text
video (Drive) -> extract + lọc frame (Drive) -> COLMAP sparse (Drive) -> train 3DGS (Drive) -> point_cloud.ply
```

**Vì sao lưu vào Google Drive thay vì `/content`:**
Colab tự ngắt session sau một khoảng idle hoặc khi hết giờ runtime, và `/content` bị xoá sạch mỗi khi session mới khởi tạo. Notebook này mount Google Drive và ghi **toàn bộ** dữ liệu trung gian (video, frame, output COLMAP, checkpoint train, model cuối) vào đó, nên:

- Không cần zip/tải xuống/tải lên thủ công giữa các bước như 2 notebook cũ.
- Nếu bị ngắt session giữa chừng, chỉ cần mount lại Drive và chạy lại từ đầu notebook — các bước đã xong sẽ tự bỏ qua (có cờ `FORCE_*` nếu muốn chạy lại ép buộc).
- Bước train có checkpoint (`--checkpoint_iterations`) lưu vào Drive, nên nếu mất kết nối giữa lúc train, lần chạy lại sẽ **resume** từ checkpoint gần nhất chứ không train lại từ 0.

**Giới hạn cần biết:** bước `colmap mapper` (bundle adjustment/SfM) luôn chạy CPU dù bật `--use_gpu` cho các bước feature extraction/matching — đây là giới hạn của chính COLMAP, không phải do notebook. Notebook này đổi sang `sequential_matcher` (thay vì `exhaustive_matcher` mặc định của `convert.py` gốc) để giảm tải bước matching, vì video quay là chuỗi frame tuần tự nên không cần so khớp mọi cặp ảnh với nhau.

## 0. Hướng dẫn quay video

- Resolution: **1080p**, FPS **30**. Có thể quay 4K nhưng notebook sẽ resize về `MAX_FRAME_WIDTH`.
- Thời lượng **20-30 giây**, đi chậm quanh vật thể đúng 1 vòng, giữ khoảng cách 40-70cm.
- Không zoom, không portrait mode, không flash. Khoá exposure/focus nếu điện thoại cho phép.
- Vật thể đứng yên, camera di chuyển quanh vật thể (không đặt điện thoại cố định rồi xoay vật thể trên bàn xoay).
- Lần đầu nên dùng vật nhỏ có texture rõ, ánh sáng đều. Tránh vật bóng, trong suốt, trắng trơn hoặc nền trắng trơn.

In [ ]:
# =============================
# 1. CONFIG - chỉnh cell này
# =============================

SCENE_NAME = "pikachu"  # đổi tên scene, vd: "cup", "statue", "mini_temple"

# Video nguồn: dùng 1 trong 2 cách
GDRIVE_URL = "https://drive.google.com/file/d/1rDQReG82d1ivA6BkDesSndsLW_HPZ46v/view?usp=sharing"  # link share Google Drive (anyone with link)
DRIVE_VIDEO_PATH = ""  # hoặc path video đã có sẵn trong Drive, vd: /content/drive/MyDrive/videos/pikachu.mp4

# Frame extraction
TARGET_SAMPLE_FPS = 4          # 4 fps x 30s = khoảng 120 frame
MAX_FRAMES = 120               # giới hạn frame để COLMAP + train nhanh trên T4
MAX_FRAME_WIDTH = 1280         # 1280 ổn cho prototype; 1600 đẹp hơn nhưng lâu hơn
MIN_BLUR_SCORE = 60.0          # thấp quá giữ nhiều frame mờ; cao quá có thể loại quá nhiều frame
JPEG_QUALITY = 95

# COLMAP
COLMAP_USE_GPU = True
CAMERA_MODEL = "OPENCV"
MATCHER = "sequential"         # "sequential" (nhanh, hợp với video quay liên tục) hoặc "exhaustive" (chậm hơn, đôi khi ổn định hơn với scene khó)
SEQUENTIAL_OVERLAP = 10        # số frame liền kề mỗi bên được đối sánh trong sequential_matcher

# Training
TRAIN_ITERS = 15000            # 7000 chỉ đủ nhanh cho test pipeline, ảnh còn mờ. 15000 nét hơn rõ rệt,
                                # đổi lại train lâu hơn (~gấp đôi so với 7000). Không khuyến khích 30000 trên T4 lần đầu.
TRAIN_RESOLUTION = 1           # 1 = dùng frame đã resize. Đổi 2 nếu bị out-of-memory.
DATA_DEVICE = "cuda"           # đổi "cpu" nếu hết VRAM lúc load ảnh
CHECKPOINT_AT = sorted(set([max(1, TRAIN_ITERS // 2), TRAIN_ITERS]))  # các mốc lưu checkpoint để resume nếu bị ngắt

# Cờ ép chạy lại (mặc định False = tự động bỏ qua bước đã xong, hỗ trợ resume sau khi bị ngắt session)
FORCE_REDOWNLOAD_VIDEO = False
FORCE_REEXTRACT_FRAMES = False
FORCE_RERUN_COLMAP = False
FORCE_RETRAIN = False

# Paths - TẤT CẢ dữ liệu trung gian nằm trong Google Drive, không dùng /content để tránh mất khi session bị ngắt
DRIVE_ROOT = "/content/drive/MyDrive/HeritageTwin"
SCENE_DIR = f"{DRIVE_ROOT}/scenes/{SCENE_NAME}"
INPUT_DIR = f"{SCENE_DIR}/input"
VIDEO_PATH = f"{SCENE_DIR}/source_video.mp4"
MODEL_DIR = f"{DRIVE_ROOT}/models/{SCENE_NAME}_3dgs_{TRAIN_ITERS}"

# Repo code + CUDA extension biên dịch: để ở /content (ephemeral), build lại mỗi session (vài phút, tránh lỗi binary
# biên dịch cũ không khớp CUDA/driver của runtime mới). Chỉ dữ liệu (video/frame/colmap/model) mới cần bền trên Drive.
REPO_DIR = "/content/gaussian-splatting"

print("Scene:", SCENE_NAME)
print("Scene dir (Drive):", SCENE_DIR)
print("Model dir (Drive):", MODEL_DIR)
print("Repo dir (ephemeral):", REPO_DIR)

In [ ]:
# =============================
# 2. Mount Google Drive (lưu trữ bền vững, sống sót qua các lần ngắt session)
# =============================
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(SCENE_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print("Đã tạo thư mục scene/model trên Drive.")

In [ ]:
# =============================
# 3. Kiểm tra GPU
# =============================
!nvidia-smi

import torch, sys
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## (Tuỳ chọn) Test nhanh bằng scene mẫu Mip-NeRF360, bỏ qua quay video + COLMAP

Chỉ chạy phần này nếu muốn kiểm tra riêng bước **train 3DGS** có chạy đúng không, trước khi tự quay video
thật. Scene "bonsai" (Mip-NeRF360 - bộ dữ liệu chuẩn mà chính bài báo 3D Gaussian Splatting dùng để đánh
giá) đã có sẵn ảnh + COLMAP sparse pose tính sẵn, nên tải xong là nhảy thẳng xuống cell **12. Train** được
luôn, bỏ qua toàn bộ mục 6-10 (lấy video, extract frame, COLMAP).

Cell dưới tải theo **range request** (chỉ lấy đúng phần scene cần trong file zip 11.67GB, không tải cả
file) - với scene "bonsai" chỉ mất khoảng 1.3GB thay vì 11.67GB.

Nếu muốn dùng scene thật của bạn, **bỏ qua cell này**, chạy tiếp từ mục 6 như bình thường.

In [ ]:
# =============================
# (Tuỳ chọn) Tải scene mẫu Mip-NeRF360 -> thẳng vào SCENE_DIR trên Drive
# =============================
import io, os, zipfile, urllib.request
from pathlib import Path

MIPNERF360_SCENE = "bonsai"  # cac scene khac: bicycle, garden, stump, treehill, flowers, room, counter, kitchen
MIPNERF360_URL = "https://storage.googleapis.com/gresearch/refraw360/360_v2.zip"


class HTTPRangeFile(io.RawIOBase):
    def __init__(self, url):
        self.url = url
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req) as resp:
            self.size = int(resp.headers["Content-Length"])
        self.pos = 0
        self.bytes_fetched = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        if whence == 0:
            self.pos = offset
        elif whence == 1:
            self.pos += offset
        elif whence == 2:
            self.pos = self.size + offset
        return self.pos

    def tell(self):
        return self.pos

    def readinto(self, b):
        n = len(b)
        if n == 0 or self.pos >= self.size:
            return 0
        end = min(self.pos + n, self.size) - 1
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        with urllib.request.urlopen(req) as resp:
            data = resp.read()
        self.bytes_fetched += len(data)
        b[: len(data)] = data
        self.pos += len(data)
        return len(data)


sparse_ok = all((Path(SCENE_DIR) / p).exists() for p in [
    "sparse/0/cameras.bin", "sparse/0/images.bin", "sparse/0/points3D.bin"
])
if sparse_ok and not FORCE_RERUN_COLMAP:
    print("SCENE_DIR đã có sẵn dữ liệu (sparse + images), bỏ qua tải scene mẫu.")
else:
    prefix = f"{MIPNERF360_SCENE}/"
    raw = HTTPRangeFile(MIPNERF360_URL)
    f = io.BufferedReader(raw, buffer_size=1024 * 1024)
    print(f"Remote zip size: {raw.size / 1024 / 1024 / 1024:.2f} GB")

    zf = zipfile.ZipFile(f)
    members = [n for n in zf.namelist() if n.startswith(prefix)]
    print(f"Tìm thấy {len(members)} file trong scene '{MIPNERF360_SCENE}'")

    Path(SCENE_DIR).mkdir(parents=True, exist_ok=True)
    for i, name in enumerate(members):
        rel = name[len(prefix):]
        # Chỉ lấy images/ (độ phân giải gốc) và sparse/ - bỏ qua images_2/4/8 (các bản resize sẵn, không cần)
        if not (rel.startswith("images/") or rel.startswith("sparse/")):
            continue
        target = Path(SCENE_DIR) / rel
        if name.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(name) as src, open(target, "wb") as dst:
            dst.write(src.read())
        if (i + 1) % 50 == 0:
            print(f"  ... {i + 1}/{len(members)}")

    print(f"Đã tải xong. Tổng dung lượng đã fetch: {raw.bytes_fetched / 1024 / 1024:.0f} MB (thay vì {raw.size / 1024 / 1024 / 1024:.2f} GB nếu tải cả file)")
    print("Scene mẫu đã sẵn sàng tại:", SCENE_DIR)
    print("Có thể bỏ qua mục 6-10, chạy thẳng xuống 'Validate dataset' và 'Train'.")

In [ ]:
# =============================
# 4. Cài system packages + python deps
# =============================
!apt-get update -qq
!apt-get install -y -qq colmap ffmpeg imagemagick

!pip install -q plyfile tqdm opencv-python gdown ninja

!colmap -h | head -n 5
!ffmpeg -version | head -n 3

In [ ]:
# =============================
# 5. Clone repo + build CUDA extensions (ephemeral, build lại mỗi session)
# =============================
import os

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive {REPO_DIR}
else:
    print("Repo đã tồn tại:", REPO_DIR)

%cd {REPO_DIR}

# Build CUDA extensions. Nếu cell này lỗi: Runtime > Restart runtime rồi chạy lại từ đầu.
# Trên Colab T4 bước này thường mất 3-8 phút.
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn

print("Install done")

In [ ]:
# =============================
# 6. Lấy video nguồn -> lưu vào Drive (SCENE_DIR/source_video.mp4)
# =============================
import os

if os.path.exists(VIDEO_PATH) and not FORCE_REDOWNLOAD_VIDEO:
    print("Video đã có sẵn trên Drive, bỏ qua bước tải xuống:", VIDEO_PATH)
else:
    if GDRIVE_URL.strip():
        import gdown
        print("Downloading from Google Drive URL...")
        gdown.download(GDRIVE_URL, VIDEO_PATH, quiet=False, fuzzy=True)
    elif DRIVE_VIDEO_PATH.strip():
        import shutil
        print("Copying video from Drive path...")
        assert os.path.exists(DRIVE_VIDEO_PATH), f"Không thấy file: {DRIVE_VIDEO_PATH}"
        shutil.copy(DRIVE_VIDEO_PATH, VIDEO_PATH)
    else:
        raise ValueError("Bạn cần điền GDRIVE_URL hoặc DRIVE_VIDEO_PATH trong cell CONFIG.")

assert os.path.exists(VIDEO_PATH), "Download/copy video thất bại."
print("Video path:", VIDEO_PATH)
print("Size MB:", os.path.getsize(VIDEO_PATH) / 1024 / 1024)

In [ ]:
# =============================
# 7. Extract frame + lọc frame mờ + resize -> lưu vào Drive (INPUT_DIR)
# =============================
import cv2, os, shutil, numpy as np
from pathlib import Path

existing_frames = list(Path(INPUT_DIR).glob("*.jpg")) if os.path.exists(INPUT_DIR) else []
if existing_frames and not FORCE_REEXTRACT_FRAMES:
    print(f"Đã có {len(existing_frames)} frame trong {INPUT_DIR}, bỏ qua bước extract.")
    print("Bật FORCE_REEXTRACT_FRAMES = True nếu muốn extract lại.")
else:
    if os.path.exists(INPUT_DIR):
        shutil.rmtree(INPUT_DIR)
    Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {VIDEO_PATH}")

    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / video_fps if video_fps else 0
    step = max(1, round(video_fps / TARGET_SAMPLE_FPS))

    print(f"Video FPS: {video_fps:.2f}")
    print(f"Total frames: {total_frames}")
    print(f"Duration: {duration:.2f}s")
    print(f"Sampling every {step} frames = {video_fps/step:.2f} fps")

    candidates = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % step == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
            candidates.append((frame_idx, blur_score, frame.copy()))
        frame_idx += 1
    cap.release()

    print("Candidate sampled frames:", len(candidates))
    if not candidates:
        raise RuntimeError("Không extract được frame nào.")

    sharp = [x for x in candidates if x[1] >= MIN_BLUR_SCORE]
    if len(sharp) < min(40, MAX_FRAMES // 2):
        print(f"WARNING: Chỉ có {len(sharp)} frame vượt blur threshold {MIN_BLUR_SCORE}. Fallback dùng top frame theo sharpness.")
        candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
        keep_pool = sorted(candidates_sorted[:max(40, min(len(candidates_sorted), MAX_FRAMES*2))], key=lambda x: x[0])
    else:
        keep_pool = sharp

    if len(keep_pool) > MAX_FRAMES:
        idxs = np.linspace(0, len(keep_pool)-1, MAX_FRAMES).round().astype(int)
        selected = [keep_pool[i] for i in idxs]
    else:
        selected = keep_pool

    print("Selected frames:", len(selected))
    print("Blur score: min/mean/max =",
          f"{min(x[1] for x in selected):.1f}",
          f"{np.mean([x[1] for x in selected]):.1f}",
          f"{max(x[1] for x in selected):.1f}")

    for out_i, (src_idx, blur, frame_bgr) in enumerate(selected):
        h, w = frame_bgr.shape[:2]
        if w > MAX_FRAME_WIDTH:
            scale = MAX_FRAME_WIDTH / w
            new_w = MAX_FRAME_WIDTH
            new_h = int(round(h * scale))
            frame_bgr = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
        out_path = f"{INPUT_DIR}/frame_{out_i:04d}.jpg"
        cv2.imwrite(out_path, frame_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])

    print("Saved frames to:", INPUT_DIR)
    print("Frame count:", len(os.listdir(INPUT_DIR)))

In [ ]:
# =============================
# 8. Preview một số frame
# =============================
import os, glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

imgs = sorted(glob.glob(f"{INPUT_DIR}/*.jpg"))
assert imgs, "Chưa có ảnh trong input."

sample = [imgs[i] for i in np.linspace(0, len(imgs)-1, min(12, len(imgs))).round().astype(int)]
plt.figure(figsize=(14, 8))
for i, p in enumerate(sample):
    im = Image.open(p)
    plt.subplot(3, 4, i+1)
    plt.imshow(im)
    plt.axis('off')
    plt.title(os.path.basename(p))
plt.tight_layout()
plt.show()

In [ ]:
# =============================
# 9. Helper: chạy subprocess với log realtime (dùng chung cho COLMAP + train)
# =============================
import subprocess, sys, time

def run_streamed(cmd, cwd=None, env=None, label=""):
    print(f"\n===== {label or ' '.join(cmd)} =====")
    print(" ".join(cmd))
    start = time.time()
    process = subprocess.Popen(
        cmd, cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="")
            sys.stdout.flush()
    except KeyboardInterrupt:
        print("\nInterrupted by user. Terminating process...")
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        raise
    ret = process.wait()
    elapsed = time.time() - start
    print(f"===== {label or cmd[0]} finished, return code {ret}, elapsed {elapsed/60:.1f} min =====")
    if ret != 0:
        raise RuntimeError(f"{label or cmd[0]} failed with return code {ret}")

In [ ]:
# =============================
# 10. Chạy COLMAP (feature_extractor -> matcher -> mapper -> undistorter) -> lưu vào Drive (SCENE_DIR)
# Bước mapper luôn chạy CPU (giới hạn của COLMAP, không tuỳ chỉnh được bằng notebook).
# Dùng sequential_matcher thay vì exhaustive_matcher của convert.py gốc vì video là chuỗi frame liên tục.
# =============================
import os
from pathlib import Path

sparse_ok = all((Path(SCENE_DIR) / p).exists() for p in [
    "sparse/0/cameras.bin", "sparse/0/images.bin", "sparse/0/points3D.bin"
])

if sparse_ok and not FORCE_RERUN_COLMAP:
    print("Đã có sẵn COLMAP sparse model trong Drive, bỏ qua bước này.")
    print("Bật FORCE_RERUN_COLMAP = True nếu muốn chạy lại.")
else:
    import shutil
    for name in ["distorted", "sparse", "database.db", "stereo", "images"]:
        p = Path(SCENE_DIR) / name
        if p.is_dir():
            shutil.rmtree(p)
        elif p.is_file():
            p.unlink()

    env = os.environ.copy()
    env.pop("QT_QPA_PLATFORM_PLUGIN_PATH", None)
    env.pop("QT_PLUGIN_PATH", None)
    env["DISPLAY"] = ":99"
    env["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"
    Path(env["XDG_RUNTIME_DIR"]).mkdir(parents=True, exist_ok=True)

    xvfb = ["xvfb-run", "-a", "-s", "-screen 0 1280x1024x24"]
    db_path = f"{SCENE_DIR}/distorted/database.db"
    Path(f"{SCENE_DIR}/distorted/sparse").mkdir(parents=True, exist_ok=True)
    use_gpu = "1" if COLMAP_USE_GPU else "0"

    feat_cmd = xvfb + [
        "colmap", "feature_extractor",
        "--database_path", db_path,
        "--image_path", INPUT_DIR,
        "--ImageReader.single_camera", "1",
        "--ImageReader.camera_model", CAMERA_MODEL,
        "--SiftExtraction.use_gpu", use_gpu,
    ]
    run_streamed(feat_cmd, env=env, label="colmap feature_extractor")

    if MATCHER == "sequential":
        match_cmd = xvfb + [
            "colmap", "sequential_matcher",
            "--database_path", db_path,
            "--SiftMatching.use_gpu", use_gpu,
            "--SequentialMatching.overlap", str(SEQUENTIAL_OVERLAP),
        ]
        run_streamed(match_cmd, env=env, label="colmap sequential_matcher")
    else:
        match_cmd = xvfb + [
            "colmap", "exhaustive_matcher",
            "--database_path", db_path,
            "--SiftMatching.use_gpu", use_gpu,
        ]
        run_streamed(match_cmd, env=env, label="colmap exhaustive_matcher")

    mapper_cmd = xvfb + [
        "colmap", "mapper",
        "--database_path", db_path,
        "--image_path", INPUT_DIR,
        "--output_path", f"{SCENE_DIR}/distorted/sparse",
        "--Mapper.ba_global_function_tolerance", "0.000001",
    ]
    run_streamed(mapper_cmd, env=env, label="colmap mapper (CPU, bước lâu nhất)")

    reconstructed = Path(f"{SCENE_DIR}/distorted/sparse/0")
    if not reconstructed.exists():
        raise RuntimeError(
            "COLMAP mapper không tạo được model '0'. Reconstruction thất bại - thử giảm MAX_FRAMES, "
            "quay lại video rõ nét hơn, hoặc đổi MATCHER = 'exhaustive'."
        )

    undistort_cmd = xvfb + [
        "colmap", "image_undistorter",
        "--image_path", INPUT_DIR,
        "--input_path", str(reconstructed),
        "--output_path", SCENE_DIR,
        "--output_type", "COLMAP",
    ]
    run_streamed(undistort_cmd, env=env, label="colmap image_undistorter")

    # image_undistorter xuất sparse/ dạng phẳng (không có /0), cần đưa vào sparse/0 để khớp format train.py cần
    flat_sparse = Path(SCENE_DIR) / "sparse"
    target_0 = flat_sparse / "0"
    if not target_0.exists():
        target_0.mkdir(parents=True, exist_ok=True)
        for item in list(flat_sparse.iterdir()):
            if item.name == "0":
                continue
            shutil.move(str(item), str(target_0 / item.name))

    print("\nCOLMAP pipeline done. Sparse model tại:", target_0)

In [ ]:
# =============================
# 11. Validate dataset cho train.py
# =============================
from pathlib import Path
import shutil

required = [
    Path(SCENE_DIR) / "sparse/0/cameras.bin",
    Path(SCENE_DIR) / "sparse/0/images.bin",
    Path(SCENE_DIR) / "sparse/0/points3D.bin",
]
print("Required sparse files:")
for p in required:
    print(p, "OK" if p.exists() else "MISSING")
if not all(p.exists() for p in required):
    raise RuntimeError("Thiếu COLMAP sparse files.")

images_dir = Path(SCENE_DIR) / "images"
if not images_dir.exists():
    print("images/ chưa có, copy từ input/ làm fallback.")
    shutil.copytree(INPUT_DIR, images_dir)

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(images_dir.glob(ext))
print("Image count in images/:", len(image_files))
if len(image_files) == 0:
    raise RuntimeError("Không có ảnh nào trong images/.")
print("Dataset OK.")

In [ ]:
# =============================
# 12. Train 3D Gaussian Splatting -> lưu vào Drive (MODEL_DIR), có checkpoint để resume
# =============================
import glob, os, sys
from pathlib import Path

%cd {REPO_DIR}

final_ply = Path(MODEL_DIR) / f"point_cloud/iteration_{TRAIN_ITERS}/point_cloud.ply"

if final_ply.exists() and not FORCE_RETRAIN:
    print("Model đã train xong trước đó, bỏ qua bước train:", final_ply)
    print("Bật FORCE_RETRAIN = True nếu muốn train lại từ đầu.")
else:
    existing_ckpts = sorted(
        glob.glob(f"{MODEL_DIR}/chkpnt*.pth"),
        key=os.path.getmtime,
    )
    start_ckpt = existing_ckpts[-1] if (existing_ckpts and not FORCE_RETRAIN) else None

    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-u", "train.py",
        "-s", SCENE_DIR,
        "-m", MODEL_DIR,
        "--iterations", str(TRAIN_ITERS),
        "--save_iterations", str(TRAIN_ITERS),
        "--checkpoint_iterations", *[str(x) for x in CHECKPOINT_AT],
        "--test_iterations", "-1",
        "--resolution", str(TRAIN_RESOLUTION),
        "--data_device", DATA_DEVICE,
        "--disable_viewer",
    ]
    if start_ckpt:
        cmd += ["--start_checkpoint", start_ckpt]
        print("Phát hiện checkpoint cũ, sẽ RESUME training từ:", start_ckpt)
    else:
        print("Không có checkpoint cũ, train từ đầu.")

    run_streamed(cmd, cwd=REPO_DIR, label="train.py")

print("MODEL_DIR:", MODEL_DIR)

In [ ]:
# =============================
# 13. Kiểm tra output + (tuỳ chọn) zip để tải về máy
# =============================
from pathlib import Path

print("Model directory:", MODEL_DIR)
!find "{MODEL_DIR}" -maxdepth 4 -type f | sed -n '1,120p'

ply_files = list(Path(MODEL_DIR).glob("point_cloud/iteration_*/point_cloud.ply"))
print("PLY files:")
for p in ply_files:
    print(p, "size MB:", p.stat().st_size / 1024 / 1024)
if not ply_files:
    raise RuntimeError("Không tìm thấy point_cloud.ply. Training có thể chưa lưu đúng.")

In [ ]:
# =============================
# 14. (Tuỳ chọn) Tải model về máy - model đã nằm sẵn trong Google Drive nên bước này chỉ để tiện lợi
# =============================
import shutil
from google.colab import files
from pathlib import Path

zip_base = f"/content/{SCENE_NAME}_gaussian_{TRAIN_ITERS}"
zip_path = shutil.make_archive(zip_base, "zip", MODEL_DIR)
print("Created:", zip_path)
print("Size MB:", Path(zip_path).stat().st_size / 1024 / 1024)
files.download(zip_path)

## Ghi chú

- Toàn bộ dữ liệu trung gian (video, frame, COLMAP sparse model, checkpoint, model 3DGS) đã nằm trong
  `MyDrive/HeritageTwin/...` - nếu Colab ngắt session, chỉ cần mount lại Drive và chạy lại notebook từ đầu,
  các bước đã xong sẽ tự động được bỏ qua.
- Bước tiếp theo: dùng viewer web tự host trong thư mục `docs/viewer/` của repo (không cần backend/database)
  để load trực tiếp `point_cloud.ply` vừa train, xem trực tiếp trên trình duyệt.